In [8]:
# Install required libraries
!pip install -q google-genai pypdf


In [9]:
import os
from google import genai

# Enter your Gemini API key here or use Colab Secrets (recommended)
# To use Colab Secrets, click the key icon on the left panel, add a secret named 'GEMINI_API_KEY'
from google.colab import userdata
try:
    API_KEY = userdata.get('GEMINI_KEY')
except:
    API_KEY = "YOUR_GEMINI_API_KEY_HERE" # Replace with your actual key if not using Secrets

os.environ["GEMINI_KEY"] = API_KEY
client = genai.Client(api_key=API_KEY)
print("API Key configured successfully!")

API Key configured successfully!


In [11]:
from google.colab import files
from pypdf import PdfReader

print("Please upload a short PDF or text document for your study session:")
uploaded = files.upload()

document_text = ""
for filename in uploaded.keys():
    if filename.endswith('.pdf'):
        reader = PdfReader(filename)
        for page in reader.pages:
            document_text += page.extract_text() + "\n"
    else:
        with open(filename, 'r', encoding='utf-8') as f:
            document_text = f.read()

print(f"\nDocument loaded successfully! Total characters: {len(document_text)}")

Please upload a short PDF or text document for your study session:


Saving RAG_study.pdf to RAG_study (1).pdf

Document loaded successfully! Total characters: 109933


In [18]:
import time
from google.genai import errors

def call_gemini(prompt):
    """Helper function with robust retries and model fallback on server load"""
    models_to_try = ['gemini-3.8-flash', 'gemini-2.5-flash']
    max_retries = 5

    for model_name in models_to_try:
        delay = 2
        for attempt in range(max_retries):
            try:
                response = client.models.generate_content(
                    model=model_name,
                    contents=prompt,
                )
                return response.text
            except Exception as e:
                # If it's a 503 (or other temporary API server error), sleep and retry
                if attempt < max_retries - 1:
                    print(f"[API Warning] {model_name} busy. Retrying in {delay} seconds...")
                    time.sleep(delay)
                    delay *= 2
                else:
                    print(f"[API Warning] {model_name} failed after {max_retries} attempts. Trying fallback model...")
                    break

    # If we get here, all models in the list failed
    raise RuntimeError("Both primary and fallback Gemini models are currently experiencing high demand. Please try running the cell again in a few moments.")

# Agent Activity Log
agent_logs = []

def log_activity(agent_name, action):
    log_entry = f"[{agent_name}] {action}"
    agent_logs.append(log_entry)
    print(f"--- {log_entry}")

print("Initializing Multi-Agent Workflow...")

Initializing Multi-Agent Workflow...


In [21]:
# --- AGENT 1: SUMMARISER ---
if 'raw_summary' not in globals() or not raw_summary:
    log_activity("Summariser Agent", "Reading document and extracting main concepts...")
    summary_prompt = f"""
    You are an expert study summariser. Read the following document and create a clear, simple summary of its main concepts.
    Rely ONLY on the provided text.

    DOCUMENT:
    {document_text}
    """
    raw_summary = call_gemini(summary_prompt)
    log_activity("Summariser Agent", "Summary generated successfully.")
else:
    log_activity("Summariser Agent", "Using existing summary to prevent redundant API calls.")

# --- AGENT 2: QUIZ CREATOR ---
if 'raw_quiz' not in globals() or not raw_quiz:
    log_activity("Quiz Creator Agent", "Drafting 5 multiple-choice questions based on the summary...")
    quiz_prompt = f"""
    You are an expert educator. Using the summary provided below, prepare 5 multiple-choice questions (MCQs) with 4 options each (A, B, C, D) and provide a clear answer key at the end.

    SUMMARY:
    {raw_summary}
    """
    raw_quiz = call_gemini(quiz_prompt)
    log_activity("Quiz Creator Agent", "Quiz and answer key drafted.")
else:
    log_activity("Quiz Creator Agent", "Using existing quiz draft to prevent redundant API calls.")

# --- AGENT 3: REVIEWER ---
if 'final_output' not in globals() or not final_output:
    log_activity("Reviewer Agent", "Checking summary and questions for accuracy and formatting...")
    review_prompt = f"""
    You are a strict QA Reviewer Agent. Review the Summary and Quiz provided below.
    1. Identify any logical inconsistencies, formatting issues, or unclear answers.
    2. Correct any issues found and output the final, polished Summary, Quiz, and Answer Key.

    SUMMARY:
    {raw_summary}

    DRAFT QUIZ:
    {raw_quiz}
    """
    final_output = call_gemini(review_prompt)
    log_activity("Reviewer Agent", "Review complete. Corrections applied and finalized.")
else:
    log_activity("Reviewer Agent", "Using existing final polished output.")

--- [Summariser Agent] Using existing summary to prevent redundant API calls.
--- [Quiz Creator Agent] Using existing quiz draft to prevent redundant API calls.
--- [Reviewer Agent] Using existing final polished output.


In [22]:
print("\n" + "="*40 + " FINAL STUDY ASSISTANT OUTPUT " + "="*40 + "\n")
print(final_output)

print("\n" + "="*40 + " AGENT ACTIVITY LOG " + "="*40 + "\n")
for log in agent_logs:
    print(log)


======================================== FINAL STUDY ASSISTANT OUTPUT ========================================

### QA Review Report

1. **Factual Accuracy & Hallucination Check:**
   * **Summary:** The draft summary faithfully and accurately captures the concepts, technical methods, taxonomy, and future directions described in the survey paper without introducing unsupported claims or external hallucinations.
   * **Quiz:** All five quiz questions directly reflect specific facts and definitions provided in the text (e.g., the definition of Small2Big chunking, FLARE confidence-based retrieval, the definition of Negative Rejection, the "Lost in the Middle" phenomenon, and the Predict module). The answer options and keys are strictly accurate.

2. **Completeness & Refinements:**
   * **RAG vs. Fine-Tuning:** Added the survey’s explicit finding that LLMs struggle to acquire new factual knowledge via unsupervised fine-tuning alone, which further cements RAG's comparative advantage.
   * *

In [27]:
print("--- QUIZ TIME ---")
print("Review the quiz questions displayed above and enter your answers below (e.g., 1: A, 2: B, 3: C, 4: D, 5: A):")
user_answers = input("\nYour Answers: ")

try:
    eval_prompt = f"""
    You are an interactive learning coach. Evaluate the learner's answers against the correct answer key in the final output. Provide a score out of 5, explain why any incorrect answers were wrong, and give encouraging, personalized feedback.

    FINAL CONTENT & ANSWER KEY:
    {final_output}

    LEARNER'S ANSWERS:
    {user_answers}
    """
    evaluation_feedback = call_gemini(eval_prompt)
    print("\n" + "="*40 + " EVALUATION & FEEDBACK " + "="*40 + "\n")
    print(evaluation_feedback)
except Exception as api_err:
    print("\n[System Notice] Gemini API is currently overloaded. Falling back to local scoring...\n")

    # Local Grading Fallback
    # Correct Answer Key: 1: B, 2: B, 3: C, 4: A, 5: B
    correct_keys = {'1': 'B', '2': 'B', '3': 'C', '4': 'A', '5': 'B'}
    explanations = {
        '1': "Small2Big uses individual sentences (small) as the retrieval unit for precision, while feeding adjacent context (big) to the LLM.",
        '2': "FLARE triggers retrieval whenever the confidence of generated terms drops below a set threshold.",
        '3': "Negative Rejection measures the model's ability to decline answering when the retrieved context is insufficient.",
        '4': "Concatenating too many documents triggers the 'lost in the middle' effect, where the LLM ignores the middle of long prompts.",
        '5': "The Predict module prompts the LLM itself to generate context directly to reduce redundancy."
    }

    # Parse user answers
    import re
    parsed_answers = {}
    pairs = re.findall(r'(\d+)\s*:\s*([A-Da-d])', user_answers)
    for num, ans in pairs:
        parsed_answers[num] = ans.upper().strip()

    score = 0
    feedback_lines = []
    for q_num, correct_ans in correct_keys.items():
        user_ans = parsed_answers.get(q_num, "No Answer Given")
        if user_ans == correct_ans:
            score += 1
            feedback_lines.append(f"Question {q_num}: Correct! (Your Answer: {user_ans})")
        else:
            feedback_lines.append(f"Question {q_num}: Incorrect. Your Answer: {user_ans}. Correct Answer: {correct_ans}\n  -> Why: {explanations[q_num]}")

    print("="*40 + " LOCAL EVALUATION " + "="*40)
    print(f"YOUR SCORE: {score} / 5\n")
    for line in feedback_lines:
        print(line)
    print("\nKeep up the study! You are doing great.")

--- QUIZ TIME ---
Review the quiz questions displayed above and enter your answers below (e.g., 1: A, 2: B, 3: C, 4: D, 5: A):

Your Answers: 1: A, 2: B, 3: C, 4: D, 5: A
[API Warning] gemini-3.8-flash busy. Retrying in 2 seconds...
[API Warning] gemini-3.8-flash busy. Retrying in 4 seconds...
[API Warning] gemini-3.8-flash busy. Retrying in 8 seconds...
[API Warning] gemini-3.8-flash busy. Retrying in 16 seconds...
[API Warning] gemini-3.8-flash failed after 5 attempts. Trying fallback model...
[API Warning] gemini-2.5-flash busy. Retrying in 2 seconds...
[API Warning] gemini-2.5-flash busy. Retrying in 4 seconds...
[API Warning] gemini-2.5-flash busy. Retrying in 8 seconds...
[API Warning] gemini-2.5-flash busy. Retrying in 16 seconds...
[API Warning] gemini-2.5-flash failed after 5 attempts. Trying fallback model...

[System Notice] Gemini API is currently overloaded. Falling back to local scoring...

======================================== LOCAL EVALUATION ========================